# [Link to GitHub Repository](https://github.com/L-nsamba/group-1-formative-3-probability-bayes-gradient-descent.git)

##1. Data handling

###Download dataset into colab
Avoids repeated upload into colab filesystem

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

dataset_path = Path("IMDB Dataset.csv")
archive_path = Path("imdb-dataset.zip")
dataset_url = "https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"

if not dataset_path.is_file():
    print("Downloading the IMDB reviews dataset...")
    urlretrieve(dataset_url, archive_path)
    with ZipFile(archive_path) as archive:
        if "IMDB Dataset.csv" not in archive.namelist():
            raise FileNotFoundError("The downloaded archive does not contain IMDB Dataset.csv")
        archive.extract("IMDB Dataset.csv", path=".")

if not dataset_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {dataset_path.resolve()}")
print(f"Successfully prepared {dataset_path}!")

## Format dataset
 . We strip punctuations and paragraph breaks

 . Only count one instance of a word per review

In [2]:
import csv

reviews = []

with open("IMDB Dataset.csv", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        text = row["review"].lower()
        text = text.replace("<br />", " ")
        # take out punctuation so "good!" counts as "good"
        for ch in ".,!?;:\"()-/*":
            text = text.replace(ch, " ")
        words = set(text.split())  # store in a set so each word counts only once per review
        reviews.append((words, row["sentiment"]))

print(f"Successfully loaded {len(reviews)} total reviews!")

Successfully loaded 50000 total reviews!


##2. Keyword Selection
4 Keywords that indicate positive sentiment: "favorite", "excellent", "wonderful", "masterpiece"

4 Keywords that indicate negative sentiment: "boring", "horrible", "worst", "sucks"

##3. Choice of conditional Probability

As a group, we chose to calculate the probability of a word indicating negative sentiment i.e P(Negative | Keyword)

In [3]:
positive_words = ["favorite", "excellent", "wonderful", "masterpiece"]
negative_words = ["horrible", "boring", "worst", "sucks"]

# 4. Define useful variables:
. total_negative_reviews

In [4]:
total = len(reviews)
total_negative_reviews = 0
for words, sentiment in reviews:
    if sentiment == "negative":
        total_negative_reviews = total_negative_reviews + 1

print("total reviews:", total)
print("negative reviews:", total_negative_reviews)

total reviews: 50000
negative reviews: 25000


##3. Bayes Theorem Implementation

Bayes Function

In [7]:
def bayes(prior, likelihood, evidence):
    return likelihood * prior / evidence

## Tying it all together

In [12]:
def negative_probability_for_keyword(word):
    with_word = 0        # reviews that have the word
    neg_with_word = 0    # negative reviews that have the word

    for words, sentiment in reviews:
        if word in words:
            with_word = with_word + 1
            if sentiment == "negative":
                neg_with_word = neg_with_word + 1

    if with_word == 0:
        return None

    prior = total_negative_reviews / total                 # P(Negative)
    likelihood = neg_with_word / total_negative_reviews    # P(word | Negative)
    marginal = with_word / total                           # P(word)
    posterior = bayes(prior, likelihood, marginal)         # P (Negative|word)
    check = neg_with_word / with_word                      # direct check

    return {
        "word": word,
        "prior": prior,
        "likelihood": likelihood,
        "marginal": marginal,
        "posterior": posterior,
        "check": check,
        "total_appearances": with_word,
        "negative_appearances": neg_with_word
    }

##4. Computation and Presentation (Table)

In [ ]:
all_words = positive_words + negative_words
results = []
for word in all_words:
    result = negative_probability_for_keyword(word)
    if result is not None:
        results.append(result)

results_by_word = {result["word"]: result for result in results}
print("Bayesian probability results (class: Negative):")
print("| keyword | prior | likelihood P(word|Negative) | marginal P(word) | posterior P(Negative|word) | direct check | reviews with keyword | negative reviews with keyword |")
print("|---------|-------|----------------------------|------------------|----------------------------|--------------|----------------------|--------------------------------|")

for word in all_words:
    result = results_by_word.get(word)
    if result is None:
        print(f"| {word} | word not found | | | | | | |")
        continue

    print(
        f"| {word} | {result['prior']:.4f} | {result['likelihood']:.4f} | "
        f"{result['marginal']:.4f} | {result['posterior']:.4f} | {result['check']:.4f} | "
        f"{result['total_appearances']} | {result['negative_appearances']} |"
    )